# NanoVLM: Accelerated Medical Diagnostics at the Edge

**Project Overview**: Adapt a Vision-Language Model (VLM) for medical diagnostics, enabling efficient inference on resource-constrained edge devices.

### Custom CUDA attention edition
This version replaces the PyTorch SDPA model backend with **NanoFlash**, a custom
**Triton CUDA** online-softmax attention implementation with its own tiled forward
and backward kernels. The JIT-compiled kernels run on the NVIDIA GPU (RTX 5070 Ti).
This is **not** simply `attn_implementation="flash_attention_2"`, and not a
wrapper that calls PyTorch SDPA during training or inference. The optional
numerical tests use SDPA *only as an independent reference*. Performance gains
are **not guaranteed**; this is an experimental learning/research kernel.

**File requirement:** keep `nanoflash_cuda.py` next to this notebook. The ZIP
package contains both files. Run the numerical parity tests before training.


In [ ]:
import os
import gc
import shutil
import subprocess
from pathlib import Path

# CUDA custom kernels are JIT compiled by Triton; no separate nvcc toolkit is required.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA unavailable in this Jupyter kernel. In WSL, run nvidia-smi, "
        "install CUDA-enabled PyTorch in .venv, select the 'NanoVLM (WSL GPU)' kernel, "
        "and restart the notebook."
    )

DEVICE = torch.device("cuda:0")
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
free_b, total_b = torch.cuda.mem_get_info(0)
print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM: {total_b / 2**30:.2f} GiB total / {free_b / 2**30:.2f} GiB free")
print("PyTorch:", torch.__version__, "PyTorch CUDA:", torch.version.cuda)
print("Training dtype:", DTYPE)
print("Running Python:", __import__("sys").executable)
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "-L"], check=False)


In [ ]:
# Dependencies are installed in the WSL .venv BEFORE launching Jupyter.
# DO NOT run Kaggle/Colab pip cells, pin Pillow<10, or change torch mid-session.
# WSL terminal:
#   python3.11 -m venv .venv && source .venv/bin/activate
#   pip install torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu128
#   pip install transformers accelerate bitsandbytes peft datasets scikit-learn triton \
#       matplotlib seaborn opencv-python-headless pillow safetensors huggingface_hub \
#       ipykernel jupyterlab
#   python -m ipykernel install --user --name nanovlm --display-name "NanoVLM (WSL GPU)"

import transformers, peft, datasets, accelerate, bitsandbytes, PIL
print("transformers:", transformers.__version__)
print("peft:", peft.__version__, "datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__, "Pillow:", PIL.__version__)


# Custom Triton CUDA kernels run under WSL, NOT on a CPU-only Jupyter kernel.
# Install the *matching CUDA-enabled PyTorch* wheel from the official PyTorch selector.
# Install triton explicitly if needed: pip install -U triton
import triton
print("Triton:", triton.__version__)
print("GPU SM capability:", torch.cuda.get_device_capability(0))


In [ ]:
# The base model and SLAKE dataset are public; login is optional for downloads.
# To publish your adapter, run `hf auth login` in the WSL terminal first.
# Alternatively, set HF_TOKEN as an environment variable BEFORE starting Jupyter.
from huggingface_hub import get_token, login

hf_token = os.environ.get("HF_TOKEN") or get_token()
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face authentication available.")
else:
    print("Continuing without a Hugging Face token (public model/dataset only).")


In [ ]:
# NanoFlash custom CUDA attention: *real GPU kernels*, not a reference wrapper.
# File nanoflash_cuda.py MUST be next to this notebook.
from pathlib import Path
if not Path("nanoflash_cuda.py").exists():
    raise FileNotFoundError("Place nanoflash_cuda.py next to the notebook (extract the complete ZIP)")
from nanoflash_cuda import (
    BACKEND_NAME, CALLS, register_nanoflash,
    numerical_self_test, flash_attention, assert_kernel_ran,
)

ATTENTION_BACKEND = register_nanoflash()
print("Registered custom CUDA attention backend:", ATTENTION_BACKEND)
print("Running forward/backward numerical parity tests against PyTorch SDPA...")
# SDPA is only used as a REFERENCE in this test. Training/inference use NanoFlash.
numerical_self_test()
print("Parity tests passed. Loading SmolVLM with custom CUDA attention next.")


### What actually changed?

NanoFlash is an **experimental custom Triton GPU implementation compiled for CUDA**.
It uses tiled attention with online softmax (FlashAttention-style) in the forward
pass, and custom recomputation-based GPU backward kernels for dQ, dK and dV.
There is **no production-path call to `torch.nn.functional.scaled_dot_product_attention`**.

The notebook registers both `AttentionInterface` **and** `AttentionMaskInterface`
so 4D causal/padding masks are preserved for the vision and language backends.
The implementation supports FP16/BF16, causal attention, GQA (grouped-query
attention), CUDA decoding with KV cache, and attention masks. It rejects unsupported
attention dropout and sliding-window patterns instead of silently changing results.

**Research limitations:** this experimental kernel may run slower than optimized
SDPA/FlashAttention. Its correctness is verified only when the GPU parity and
training smoke tests finish successfully on your own machine. CUDA graph capture,
custom weight biases, variable-length packed attention, dropout > 0, and large
scale performance optimization are not provided. The input 4D attention masks
can still consume quadratic memory even though the kernel itself does not store
quadratic attention scores or probabilities.


In [ ]:
from datasets import load_dataset
from io import BytesIO
from PIL import Image

splits = load_dataset("mdwiratathya/SLAKE-vqa-english")
train_dataset = splits["train"]
val_dataset = splits["validation"]
test_dataset = splits["test"]

# Actual image content from SLAKE. Never substitute fabricated blank images.
def read_medical_image(dataset, idx):
    raw = dataset.data["image"][idx].as_py()
    if raw.get("bytes"):
        return Image.open(BytesIO(raw["bytes"])).convert("RGB")
    path = raw.get("path")
    if path and Path(path).is_file():
        return Image.open(path).convert("RGB")
    raise ValueError(f"Image missing at index {idx}; refusing to substitute synthetic data")

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

all_answers = [str(a) for a in train_dataset["answer"]]
all_questions = [str(q) for q in train_dataset["question"]]
print("Training Q&A pairs:", len(train_dataset))
print("Mean question words:", round(np.mean([len(q.split()) for q in all_questions]), 2))
print("Mean answer words:", round(np.mean([len(a.split()) for a in all_answers]), 2))
print("Unique answers:", len(set(all_answers)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, idx in zip(axes.flat, random.sample(range(len(train_dataset)), 6)):
    image = read_medical_image(train_dataset, idx)
    ax.imshow(image)
    ax.set_title(f"Q: {all_questions[idx][:55]}\nA: {all_answers[idx][:40]}", fontsize=8)
    ax.axis("off")
plt.suptitle("SLAKE — real medical images")
plt.tight_layout()
plt.show()


In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=DTYPE,
    bnb_4bit_use_double_quant=True,
)

# Keep official image processing defaults; don't force a size that may be ignored.
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},  # load on your NVIDIA GPU, not silently on CPU
    dtype=DTYPE,
    attn_implementation=ATTENTION_BACKEND,
)
print("Model:", MODEL_ID)
print("CUDA allocation:", round(torch.cuda.memory_allocated() / 2**30, 3), "GiB")
print("Device map:", getattr(model, "hf_device_map", "unknown"))


# Ensure BOTH image encoder and text decoder retained the custom backend.
for component, cfg in (("Vision encoder", model.config.vision_config),
                       ("Language decoder", model.config.text_config)):
    active = getattr(cfg, "_attn_implementation", "unknown")
    print(f"{component} backend:", active)
    if active != ATTENTION_BACKEND:
        raise RuntimeError(f"{component} did not select NanoFlash ({active})")


In [ ]:
def build_messages(question):
    # SmolVLM-500M's chat template expects content as a list of typed elements.
    # Put instructions in the user TEXT element, NOT a plain-string system message.
    q = str(question)
    ql = q.lower()
    binary = any(t in ql for t in ["is there", "does ", "do ", "are there", "has ", "have "])
    hint = "Answer Yes or No only." if binary else "Answer briefly and precisely."
    return [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": f"{hint}\nQuestion: {q}"},
        ],
    }]


def run_inference(model, processor, image, question, max_new_tokens=32):
    prompt = processor.apply_chat_template(build_messages(question), add_generation_prompt=True)
    inputs = processor(text=prompt, images=[image], return_tensors="pt").to(DEVICE)
    end_token = processor.tokenizer.convert_tokens_to_ids("<end_of_utterance>")
    if end_token is None or end_token == processor.tokenizer.unk_token_id:
        end_token = processor.tokenizer.eos_token_id

    with torch.inference_mode():
        ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=end_token,
            pad_token_id=(processor.tokenizer.pad_token_id if processor.tokenizer.pad_token_id is not None
                          else processor.tokenizer.eos_token_id),
        )
    completion = ids[:, inputs["input_ids"].shape[-1]:]
    return processor.batch_decode(completion, skip_special_tokens=True)[0].strip()

print("Zero-shot predictions before fine-tuning:")
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, idx in zip(axes, random.sample(range(len(test_dataset)), 4)):
    img = read_medical_image(test_dataset, idx)
    q, gt = str(test_dataset.data["question"][idx].as_py()), str(test_dataset.data["answer"][idx].as_py())
    pred = run_inference(model, processor, img, q)
    ax.imshow(img)
    ax.set_title(f"Q: {q[:45]}\nGT: {gt[:40]}\nPred: {pred[:40]}", fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Note: don't call unload() on a freshly loaded base model.
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable LoRA parameters: {trainable:,} / {total:,} ({trainable/total:.2%})")


In [ ]:
from torch.utils.data import Dataset

class MedVQADataset(Dataset):
    """Real SLAKE examples with assistant-only supervised loss."""
    def __init__(self, data, processor):
        self.dataset = data
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image = read_medical_image(self.dataset, idx)
        row = self.dataset.data
        question = str(row["question"][idx].as_py())
        answer = str(row["answer"][idx].as_py())
        # NO random left/right flips or color jitter: medical anatomy is direction-sensitive.
        prompt = self.processor.apply_chat_template(
            build_messages(question), add_generation_prompt=True
        )
        full_text = prompt + answer + "<end_of_utterance>"
        prompt_enc = self.processor(text=prompt, images=[image], return_tensors="pt")
        full_enc = self.processor(text=full_text, images=[image], return_tensors="pt")
        ids = full_enc["input_ids"].squeeze(0)
        pids = prompt_enc["input_ids"].squeeze(0)

        # At a tokenization boundary the tokenizer can merge nearby text; measure
        # the common prefix instead of blindly assuming the prompt length matches.
        common_len = 0
        for a, b in zip(pids.tolist(), ids.tolist()):
            if a != b:
                break
            common_len += 1
        labels = ids.clone()
        labels[:common_len] = -100
        if not (labels != -100).any():
            raise RuntimeError(f"No assistant answer tokens to train on at index {idx}")

        features = {
            "input_ids": ids,
            "attention_mask": full_enc["attention_mask"].squeeze(0),
            "labels": labels,
            "pixel_values": full_enc["pixel_values"].squeeze(0),
        }
        if "pixel_attention_mask" in full_enc:
            features["pixel_attention_mask"] = full_enc["pixel_attention_mask"].squeeze(0)
        return features

train_ds = MedVQADataset(train_dataset, processor)
val_ds = MedVQADataset(val_dataset, processor)
sample = train_ds[0]
print("Training / validation examples:", len(train_ds), len(val_ds))
print("input_ids:", sample["input_ids"].shape)
print("pixel_values (num_images, 3, H, W):", sample["pixel_values"].shape)
print("Supervised answer tokens:", int((sample["labels"] != -100).sum()))


In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from torch.nn.utils.rnn import pad_sequence

OUTPUT_DIR = "./nanovlm-medical-checkpoints"
# Set to 1 if you run out of memory; increase gradient_accumulation_steps accordingly.
MICRO_BATCH = 2
ACCUM_STEPS = 4

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=MICRO_BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=ACCUM_STEPS,
    learning_rate=1e-4,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,
    optim="paged_adamw_8bit",
    fp16=(DTYPE == torch.float16),
    bf16=(DTYPE == torch.bfloat16),
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_pin_memory=True,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    max_grad_norm=1.0,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=42,
)


def collate_fn(examples):
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is None:
        pad_id = processor.tokenizer.eos_token_id
    batch = {
        "input_ids": pad_sequence([x["input_ids"] for x in examples], batch_first=True, padding_value=pad_id),
        "attention_mask": pad_sequence([x["attention_mask"] for x in examples], batch_first=True, padding_value=0),
        "labels": pad_sequence([x["labels"] for x in examples], batch_first=True, padding_value=-100),
    }
    # SmolVLM expects (batch, num_images, channels, height, width).
    shapes = [x["pixel_values"].shape for x in examples]
    n = max(s[0] for s in shapes)
    c = shapes[0][1]
    h = max(s[2] for s in shapes)
    w = max(s[3] for s in shapes)
    pixel_values = torch.zeros((len(examples), n, c, h, w), dtype=examples[0]["pixel_values"].dtype)
    pixel_attention_mask = torch.zeros((len(examples), n, h, w), dtype=torch.long)
    for i, example in enumerate(examples):
        p = example["pixel_values"]
        ni, _, hi, wi = p.shape
        pixel_values[i, :ni, :, :hi, :wi] = p
        if "pixel_attention_mask" in example:
            pixel_attention_mask[i, :ni, :hi, :wi] = example["pixel_attention_mask"]
        else:
            pixel_attention_mask[i, :ni, :hi, :wi] = 1
    batch["pixel_values"] = pixel_values
    batch["pixel_attention_mask"] = pixel_attention_mask
    return batch

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

# One-sample forward/backward smoke test
first_batch = collate_fn([train_ds[0]])
first_batch = {
    k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v
    for k, v in first_batch.items()
}

model.train()
model.zero_grad(set_to_none=True)

# Forward pass using mixed precision
with torch.autocast(device_type="cuda", dtype=DTYPE):
    smoke_loss = model(**first_batch).loss

print("One-sample forward loss:", smoke_loss.detach().item())

assert torch.isfinite(smoke_loss).item(), "Loss is NaN/Inf"

# Backward pass — execute EXACTLY ONCE
smoke_loss.backward()

# Verify custom CUDA kernel execution
assert_kernel_ran(min_forward=1, min_backward=1)

assert CALLS["vision"] > 0 and CALLS["decoder"] > 0, (
    f"Custom backend was not called by both towers: {CALLS}"
)

print("Custom CUDA FlashAttention smoke test passed!")

# Clean up gradients before training
model.zero_grad(set_to_none=True)
del first_batch, smoke_loss
gc.collect()
torch.cuda.empty_cache()
print("Trainer ready. Effective batch size:", MICRO_BATCH * ACCUM_STEPS)
print("Attention execution summary:", CALLS)


In [ ]:
print("Starting local GPU QLoRA fine-tuning...")
model.config.use_cache = False
train_result = trainer.train()  # To resume: trainer.train(resume_from_checkpoint=True)
print("Training complete:", train_result.metrics)
print("Peak GPU allocated:", round(torch.cuda.max_memory_allocated() / 2**30, 3), "GiB")

assert_kernel_ran(min_forward=1, min_backward=1)
print("Training used custom CUDA attention:", CALLS)


In [ ]:
# ── Plot Training Curves ──
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_losses = [(h["step"], h["loss"]) for h in log_history if "loss" in h and "eval_loss" not in h]
eval_losses  = [(h["step"], h["eval_loss"]) for h in log_history if "eval_loss" in h]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

if train_losses:
    steps, losses = zip(*train_losses)
    ax1.plot(steps, losses, "b-", alpha=0.7, label="Train Loss")
    ax1.set_xlabel("Step")
    ax1.set_ylabel("Loss")
    ax1.set_title("Training Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

if eval_losses:
    steps, losses = zip(*eval_losses)
    ax2.plot(steps, losses, "r-o", label="Eval Loss")
    ax2.set_xlabel("Step")
    ax2.set_ylabel("Loss")
    ax2.set_title("Validation Loss")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

plt.suptitle("NanoVLM Training Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
ADAPTER_DIR = "./nanovlm-medic-adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)
adapter_size = sum(p.stat().st_size for p in Path(ADAPTER_DIR).rglob("*") if p.is_file()) / 2**20
print("Saved local adapter:", Path(ADAPTER_DIR).resolve())
print("Adapter and processor files:", round(adapter_size, 2), "MiB")


In [ ]:
import re
import time
from collections import Counter

def normalize_answer(text):
    text = re.sub(r"[^a-z0-9\s]", " ", str(text).lower())
    text = " ".join(text.split())
    aliases = {
        "computed tomography": "ct", "x ray": "xray", "magnetic resonance imaging": "mri",
        "radiograph": "xray", "thorax": "chest", "thoracic": "chest",
        "hepatic": "liver", "pulmonary": "lung", "renal": "kidney",
    }
    return aliases.get(text, text)

def compute_token_f1(p, g):
    a, b = normalize_answer(p).split(), normalize_answer(g).split()
    if not a or not b:
        return float(a == b)
    common = sum((Counter(a) & Counter(b)).values())
    if common == 0:
        return 0.0
    precision, recall = common / len(a), common / len(b)
    return 2 * precision * recall / (precision + recall)

model = trainer.model
model.eval()
model.config.use_cache = True
predictions, ground_truths, inference_times = [], [], []

for idx in range(len(test_dataset)):
    image = read_medical_image(test_dataset, idx)
    question = str(test_dataset.data["question"][idx].as_py())
    answer = str(test_dataset.data["answer"][idx].as_py())
    torch.cuda.synchronize()
    start = time.perf_counter()
    prediction = run_inference(model, processor, image, question)
    torch.cuda.synchronize()
    inference_times.append(time.perf_counter() - start)
    predictions.append(prediction)
    ground_truths.append(answer)
    if (idx + 1) % 100 == 0:
        print(f"Evaluated {idx + 1}/{len(test_dataset)}")

em_score = np.mean([normalize_answer(p) == normalize_answer(g)
                    for p, g in zip(predictions, ground_truths)])
avg_f1 = np.mean([compute_token_f1(p, g) for p, g in zip(predictions, ground_truths)])
avg_latency = float(np.mean(inference_times)) * 1000
throughput = 1000.0 / avg_latency
print(f"Test EM: {em_score:.4f} | Test token F1: {avg_f1:.4f}")
print(f"Latency: {avg_latency:.1f} ms/image | Sequential throughput: {throughput:.2f} images/s")


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, idx in zip(axes.flat, random.sample(range(len(test_dataset)), 8)):
    image = read_medical_image(test_dataset, idx)
    q = str(test_dataset.data["question"][idx].as_py())
    gt, pred = ground_truths[idx], predictions[idx]
    correct = normalize_answer(gt) == normalize_answer(pred)
    ax.imshow(image)
    ax.set_title(f"Q: {q[:54]}\nGT: {gt[:35]}\nPred: {pred[:35]}",
                 fontsize=8, color="green" if correct else "red")
    ax.axis("off")
plt.suptitle("Fine-tuned predictions — SLAKE held-out test split")
plt.tight_layout(); plt.show()


---
## Part 8: Local GPU Latency and Memory Profiling
These are RTX GPU measurements, not real Jetson/Raspberry Pi hardware measurements.


In [ ]:
print("Local RTX GPU inference profile (NOT a real embedded-device benchmark)")
model.eval()
image = read_medical_image(test_dataset, 0)
question = str(test_dataset.data["question"][0].as_py())
for _ in range(3):
    _ = run_inference(model, processor, image, question)

latencies = []
torch.cuda.reset_peak_memory_stats()
for idx in range(min(50, len(test_dataset))):
    image = read_medical_image(test_dataset, idx)
    question = str(test_dataset.data["question"][idx].as_py())
    torch.cuda.synchronize()
    start = time.perf_counter()
    _ = run_inference(model, processor, image, question)
    torch.cuda.synchronize()
    latencies.append((time.perf_counter() - start) * 1000)

peak_mem = torch.cuda.max_memory_allocated() / 2**30
print(f"Mean: {np.mean(latencies):.1f} ms, median: {np.median(latencies):.1f} ms")
print(f"P95: {np.percentile(latencies, 95):.1f} ms, peak PyTorch GPU allocation: {peak_mem:.2f} GiB")
print("For actual edge claims, also benchmark on the target edge hardware.")
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(latencies, bins=15)
ax.set(title="Sequential local GPU latency", xlabel="Milliseconds", ylabel="Number of cases")
plt.show()


# OPTIONAL kernel-only comparison on identical tensors (not end-to-end model speed).
# Native SDPA is ONLY the benchmark baseline, not used in model training/inference.
from torch.nn.functional import scaled_dot_product_attention as _sdpa_reference
from triton.testing import do_bench
bench_q = torch.randn((1, 8, 256, 64), device=DEVICE, dtype=DTYPE)
bench_k = torch.randn_like(bench_q)
bench_v = torch.randn_like(bench_q)
custom_ms = do_bench(lambda: flash_attention(bench_q, bench_k, bench_v, is_causal=True))
reference_ms = do_bench(lambda: _sdpa_reference(bench_q, bench_k, bench_v, is_causal=True))
print(f"Kernel-only forward latency: custom NanoFlash={custom_ms:.3f}ms, "
      f"native SDPA={reference_ms:.3f}ms")
print("Speedups, if any, must be measured: hand-written kernels are not necessarily faster.")


---
## Part 9: Interactive Demo — Medical Diagnostic Assistant

In [ ]:
def medical_diagnosis_demo(model, processor, image, questions, title="Medical Image"):
    """Educational medical VQA demo; not a clinical diagnostic system."""
    print("Educational VQA demo:", title)
    results = []
    for question in questions:
        torch.cuda.synchronize()
        start = time.perf_counter()
        answer = run_inference(model, processor, image, question)
        torch.cuda.synchronize()
        ms = (time.perf_counter() - start) * 1000
        print(f"Q: {question}\nA: {answer} ({ms:.0f} ms)\n")
        results.append({"question": question, "answer": answer, "latency_ms": ms})
    return results

clinical_questions = [
    "What modality is used to take this image?",
    "Which part of the body does this image belong to?",
    "What is the main organ in the image?",
    "Does the image show any abnormality?",
]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, idx in zip(axes, random.sample(range(len(test_dataset)), 3)):
    image = read_medical_image(test_dataset, idx)
    ax.imshow(image); ax.axis("off")
    medical_diagnosis_demo(model, processor, image, clinical_questions, title=f"SLAKE index {idx}")
plt.tight_layout(); plt.show()


---
## Part 10: Model Card & Summary Dashboard

In [ ]:
# ── NanoVLM Summary Dashboard ──
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
# ── Extract metric variables from Cell [18] output lists ──
import numpy as np

def _normalize(text):
    return normalize_answer(text)

def _token_f1(pred, gt):
    pred_tokens = _normalize(pred).split()
    gt_tokens   = _normalize(gt).split()
    if not pred_tokens or not gt_tokens:
        return float(_normalize(pred) == _normalize(gt))
    common = sum(min(pred_tokens.count(t), gt_tokens.count(t)) for t in set(pred_tokens))
    if common == 0:
        return 0.0
    precision = common / len(pred_tokens)
    recall    = common / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)

# Exact Match and Token F1 — derived from Cell [18] lists
exact_match  = sum(
    _normalize(p) == _normalize(g)
    for p, g in zip(predictions, ground_truths)
) / len(ground_truths)

token_f1 = sum(
    compute_token_f1(p, g)
    for p, g in zip(predictions, ground_truths)
) / len(ground_truths)

# Latency and throughput — derived from Cell [18] inference_times list
avg_latency_ms     = float(np.mean(inference_times)) * 1000   # convert s → ms
throughput_per_sec = 1.0 / float(np.mean(inference_times))

# GPU memory at inference time — read live from CUDA
gpu_mem_gb = torch.cuda.memory_allocated() / 1024**3

# ── Now the rest of the dashboard plotting code follows ──

# ── Pull live data from notebook variables ──
log_history = trainer.state.log_history

train_steps  = [h["step"] for h in log_history if "loss" in h and "eval_loss" not in h]
train_losses = [h["loss"] for h in log_history if "loss" in h and "eval_loss" not in h]
eval_steps   = [h["step"] for h in log_history if "eval_loss" in h]
eval_losses  = [h["eval_loss"] for h in log_history if "eval_loss" in h]

# Epoch-level metrics derived from actual trainer log history.
eval_records = [h for h in log_history if "eval_loss" in h]
epoch_nums = [int(round(h.get("epoch", i+1))) for i, h in enumerate(eval_records)]
epoch_val = [float(h["eval_loss"]) for h in eval_records]
epoch_train = []
prev_epoch = 0.0
for record in eval_records:
    curr = float(record.get("epoch", prev_epoch + 1))
    losses = [float(h["loss"]) for h in log_history
              if "loss" in h and "eval_loss" not in h
              and prev_epoch < float(h.get("epoch", 0)) <= curr]
    epoch_train.append(float(np.mean(losses)) if losses else float("nan"))
    prev_epoch = curr

# ── Style ──
plt.rcParams.update({
    "figure.facecolor":  "#0f0f0f",
    "axes.facecolor":    "#1a1a1a",
    "axes.edgecolor":    "#333333",
    "axes.labelcolor":   "#cccccc",
    "xtick.color":       "#888888",
    "ytick.color":       "#888888",
    "text.color":        "#eeeeee",
    "grid.color":        "#2a2a2a",
    "grid.linewidth":    0.8,
    "font.family":       "monospace",
})

TEAL   = "#4f98a3"
CORAL  = "#dd6974"
GOLD   = "#e8af34"
GREEN  = "#6daa45"
MUTED  = "#555555"
WHITE  = "#eeeeee"

fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor("#0f0f0f")

gs = gridspec.GridSpec(
    3, 4,
    figure=fig,
    hspace=0.55,
    wspace=0.38,
    left=0.06, right=0.97,
    top=0.91, bottom=0.06
)

# ── Title ──
fig.text(
    0.5, 0.965,
    "NanoVLM  ·  Medical Diagnostics at the Edge  ·  SmolVLM-500M + QLoRA",
    ha="center", va="center",
    fontsize=15, fontweight="bold", color=WHITE,
    fontfamily="monospace"
)
fig.text(
    0.5, 0.942,
    f"SLAKE Medical VQA  ·  {len(epoch_nums)} Eval Epochs  ·  r=8  ·  4-bit NF4  ·  SDPA",
    ha="center", va="center",
    fontsize=9, color="#888888", fontfamily="monospace"
)

# ─────────────────────────────────────────────
# [0,0] Training Loss (step-level)
# ─────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(train_steps, train_losses, color=TEAL, linewidth=1.2, alpha=0.85, label="Train Loss")
ax1.fill_between(train_steps, train_losses, alpha=0.08, color=TEAL)
ax1.set_title("Training Loss  (step-level)", color=WHITE, fontsize=10, pad=8)
ax1.set_xlabel("Step", fontsize=8)
ax1.set_ylabel("Loss", fontsize=8)
ax1.legend(fontsize=8, framealpha=0.15)
ax1.grid(True)

# ─────────────────────────────────────────────
# [0,2] Validation Loss (epoch checkpoints)
# ─────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2:])
ax2.plot(eval_steps, eval_losses, color=CORAL, linewidth=2, marker="o",
         markersize=7, label="Val Loss")
ax2.fill_between(eval_steps, eval_losses, alpha=0.08, color=CORAL)
ax2.set_title("Validation Loss  (epoch checkpoints)", color=WHITE, fontsize=10, pad=8)
ax2.set_xlabel("Step", fontsize=8)
ax2.set_ylabel("Loss", fontsize=8)
ax2.legend(fontsize=8, framealpha=0.15)
ax2.grid(True)

# ─────────────────────────────────────────────
# [1,0–1] Epoch Comparison Bar Chart
# ─────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, :2])
x      = np.arange(len(epoch_nums))
width  = 0.35
bars1  = ax3.bar(x - width/2, epoch_train, width, label="Train Loss", color=TEAL,   alpha=0.85, zorder=3)
bars2  = ax3.bar(x + width/2, epoch_val,   width, label="Val Loss",   color=CORAL,  alpha=0.85, zorder=3)

for bar in bars1:
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f"{bar.get_height():.3f}", ha="center", va="bottom",
             fontsize=7.5, color=TEAL)
for bar in bars2:
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f"{bar.get_height():.3f}", ha="center", va="bottom",
             fontsize=7.5, color=CORAL)

ax3.set_title("Train vs Validation Loss  per Epoch", color=WHITE, fontsize=10, pad=8)
ax3.set_xticks(x)
ax3.set_xticklabels([f"Epoch {e}" for e in epoch_nums], fontsize=8)
ax3.set_ylabel("Loss", fontsize=8)
ax3.legend(fontsize=8, framealpha=0.15)
ax3.grid(True, axis="y")

# ─────────────────────────────────────────────
# [1,2] Accuracy KPI Bars
# ─────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
metrics      = ["Exact Match", "Token F1"]
values       = [exact_match, token_f1]
colors_kpi   = [GREEN, GOLD]
bars         = ax4.barh(metrics, values, color=colors_kpi, alpha=0.9, height=0.4)
ax4.set_xlim(0, 1.0)
ax4.axvline(x=0.70, color=MUTED, linestyle="--", linewidth=0.8, alpha=0.6)
for bar, val in zip(bars, values):
    ax4.text(val + 0.01, bar.get_y() + bar.get_height()/2,
             f"{val:.4f}", va="center", fontsize=10, fontweight="bold",
             color=WHITE)
ax4.set_title("Accuracy Metrics", color=WHITE, fontsize=10, pad=8)
ax4.set_xlabel("Score  (0–1)", fontsize=8)
ax4.grid(True, axis="x")

# ─────────────────────────────────────────────
# [1,3] Edge Performance KPIs (tiles)
# ─────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 3])
ax5.set_xlim(0, 1); ax5.set_ylim(0, 1)
ax5.axis("off")
ax5.set_title("Local GPU Inference Profile", color=WHITE, fontsize=10, pad=8)

kpis = [
    ("Latency",     f"{avg_latency_ms:.1f} ms",  TEAL,  0.78),
    ("Throughput",  f"{throughput_per_sec:.1f} samples/s", GOLD, 0.50),
    ("GPU Memory",  f"{gpu_mem_gb:.2f} GB",       GREEN, 0.22),
]
for label, val, clr, y in kpis:
    rect = FancyBboxPatch((0.04, y - 0.10), 0.92, 0.22,
                          boxstyle="round,pad=0.015",
                          facecolor=clr, alpha=0.12,
                          edgecolor=clr, linewidth=1.0)
    ax5.add_patch(rect)
    ax5.text(0.50, y + 0.03, val,   ha="center", va="center",
             fontsize=13, fontweight="bold", color=clr)
    ax5.text(0.50, y - 0.05, label, ha="center", va="center",
             fontsize=7.5, color="#aaaaaa")

# ─────────────────────────────────────────────
# [2,0–1] Model Config Table
# ─────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, :2])
ax6.axis("off")
ax6.set_title("Model & Training Configuration", color=WHITE, fontsize=10, pad=8)

trainable_pct = 100 * trainable / total
config_data = [
    ["Base Model",          "SmolVLM-500M-Instruct"],
    ["Quantization",        "4-bit NF4 + double quant"],
    ["Attention",           "Custom CUDA Attention"],
    ["LoRA Rank / Alpha",   f"r={lora_config.r}  /  α={lora_config.lora_alpha}"],
    ["LoRA Dropout",        f"{lora_config.lora_dropout}"],
    ["Trainable Params",    f"{trainable/1e6:.1f}M  ({trainable_pct:.2f}%)"],
    ["Adapter File Size",   f"{adapter_size:.1f} MB"],
    ["Image Resolution",    f"{sample['pixel_values'].shape[-1]}px model input"],
    ["Epochs / LR",         f"{training_args.num_train_epochs:g}  /  {training_args.learning_rate:g} (cosine)"],
    ["Optimizer",           "PagedAdamW 8-bit"],
]
col_labels = ["Parameter", "Value"]
table = ax6.table(
    cellText=config_data,
    colLabels=col_labels,
    cellLoc="left",
    loc="center",
    bbox=[0, -0.05, 1, 1.1]
)
table.auto_set_font_size(False)
table.set_fontsize(8.5)
for (row, col), cell in table.get_celld().items():
    cell.set_facecolor("#1a1a1a" if row % 2 == 0 else "#222222")
    cell.set_edgecolor("#333333")
    cell.set_text_props(color=WHITE if row > 0 else TEAL)
    if row == 0:
        cell.set_facecolor("#0a2e33")
        cell.set_text_props(color=TEAL, fontweight="bold")

# ─────────────────────────────────────────────
# [2,2–3] Dataset Summary + Training Stats
# ─────────────────────────────────────────────
ax7 = fig.add_subplot(gs[2, 2:])
ax7.axis("off")
ax7.set_title("Dataset & Training Summary", color=WHITE, fontsize=10, pad=8)

summary_data = [
    ["Dataset",               "SLAKE Medical VQA (English)"],
    ["Train / Val / Test",    "4,919  /  1,053  /  1,061"],
    ["Unique Answers",        "227  (avg 1.4 words)"],
    ["Logged Train Steps",    f"{max(train_steps) if train_steps else 0}"],
    ["Training Time",         f"{train_result.metrics['train_runtime']/3600:.2f} hrs"],
    ["Samples / Second",      f"{train_result.metrics['train_samples_per_second']:.2f}"],
    ["Peak GPU Memory",       f"{torch.cuda.max_memory_allocated()/1024**3:.2f} GB"],
    ["Final Train Loss",      f"{train_result.metrics['train_loss']:.4f}"],
    ["Best Val Loss",         f"{min(eval_losses):.4f}  (epoch {eval_losses.index(min(eval_losses))+1})"],
    ["Test EM  /  F1",        f"{exact_match:.4f}  /  {token_f1:.4f}"],
]
table2 = ax7.table(
    cellText=summary_data,
    colLabels=["Metric", "Value"],
    cellLoc="left",
    loc="center",
    bbox=[0, -0.05, 1, 1.1]
)
table2.auto_set_font_size(False)
table2.set_fontsize(8.5)
for (row, col), cell in table2.get_celld().items():
    cell.set_facecolor("#1a1a1a" if row % 2 == 0 else "#222222")
    cell.set_edgecolor("#333333")
    cell.set_text_props(color=WHITE if row > 0 else CORAL)
    if row == 0:
        cell.set_facecolor("#33100f")
        cell.set_text_props(color=CORAL, fontweight="bold")

plt.savefig("nanovlm_dashboard.png", dpi=150, bbox_inches="tight",
            facecolor="#0f0f0f")
plt.show()
print("Dashboard saved → nanovlm_dashboard.png")

---
## Part 11: Push Model to Hugging Face Hub (Optional)

In [ ]:
# Default OFF: this notebook should never publish without your explicit choice.
PUSH_TO_HUB = True
HUB_REPO = "Saspatez/nanovlm-medic-slake"
if PUSH_TO_HUB:
    if HUB_REPO.startswith("YOUR_USERNAME/"):
        raise ValueError("Replace YOUR_USERNAME before uploading")
    # Publish the saved adapter, not an accidentally merged or different model.
    model.push_to_hub(HUB_REPO)
    processor.push_to_hub(HUB_REPO)
    print("Published adapter:", f"https://huggingface.co/{HUB_REPO}")
else:
    print("Skipping optional Hugging Face Hub upload; saved adapter remains local.")


---
## Part 12: Reload & Inference (Verify Saved Adapter)

In [ ]:
from peft import PeftModel

print("Reloading local adapter:", ADAPTER_DIR)
# Free the previous training model to leave room for a second copy.
del trainer, model
import gc
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    dtype=DTYPE,
    attn_implementation=ATTENTION_BACKEND,
)
model_reloaded = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model_reloaded.eval()
processor_reloaded = AutoProcessor.from_pretrained(ADAPTER_DIR)
idx = 5
image = read_medical_image(test_dataset, idx)
question = str(test_dataset.data["question"][idx].as_py())
answer_gt = str(test_dataset.data["answer"][idx].as_py())
answer_pred = run_inference(model_reloaded, processor_reloaded, image, question)
print("Question:", question, " | Ground truth:", answer_gt, " | Prediction:", answer_pred)
print("Adapter reload successful.")

assert_kernel_ran(min_forward=1, min_backward=1)
print("Reinitialized model still uses:", model_reloaded.config._attn_implementation)
